# Hard Negative 정상 데이터 생성

KoELECTRA가 오탐하는 케이스 — 피싱 키워드(주민번호, 이체, 비밀번호 등)가 등장하지만  
실제로는 완전히 정상인 대화 — 를 GPT-4o-mini로 생성합니다.

**생성 목표: 1,500개 (label=[0,0,0])**

| 그룹 | 시나리오 | 목표 |
|------|----------|------|
| A | 주민번호가 나오는 정상 본인확인 (합격자, 병원, 통신사 등) | 200개 |
| B | 이체 + 비밀번호가 나오는 정상 인터넷뱅킹 문의 | 200개 |
| C | 계좌/카드 관련 정상 금융 문의 | 200개 |
| D | 대출/상환 관련 정상 금융 상담 | 200개 |
| E | 인증번호 — 카카오/네이버 로그인, 앱 가입, 본인인증 | 150개 |
| F | 계좌번호 — 중고거래 환불, 더치페이, 택배 환불 | 150개 |
| G | 카드번호 + 유효기간 — 전화 주문 결제, 호텔 예약 | 100개 |
| H | 신분증 + OTP — 비대면 서비스, 은행 앱 오류 문의 | 150개 |
| I | 명의 + 환급 — 차량/부동산 명의 이전, 세금/보험 환급 | 150개 |

**드라이브 출력 경로:**
```
MyDrive/VoicePhishingData/data/raw/normal_hard_negative.json
```

생성 후 `train_colab.ipynb`의 `SOURCE_FILES`에 이 파일을 추가하고 재학습하세요.

In [ ]:
# ── 셀 1: 드라이브 마운트 & 패키지 설치 ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install openai tqdm -q

In [ ]:
# ── 셀 2: 전체 스크립트 ───────────────────────────────────────
import json
import os
import time
from pathlib import Path
from tqdm import tqdm

try:
    from openai import OpenAI
except ImportError:
    os.system("pip install openai -q")
    from openai import OpenAI

# ── 경로 설정 ─────────────────────────────────────────────────
DRIVE_BASE  = Path("/content/drive/MyDrive/VoicePhishingData")
OUTPUT_PATH = DRIVE_BASE / "data/raw/normal_hard_negative.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── 생성 설정 ─────────────────────────────────────────────────
MODEL       = "gpt-4o-mini"
BATCH_SIZE  = 10
SLEEP_SEC   = 0.5
RETRY_LIMIT = 3

# 그룹별 목표 개수
QUOTA = {
    "A_identity":    200,   # 주민번호/신분 확인 (정상)
    "B_banking":     200,   # 이체/비밀번호 (정상 인터넷뱅킹)
    "C_card":        200,   # 계좌/카드 (정상 금융 문의)
    "D_loan":        200,   # 대출/상환 (정상 금융 상담)
    "E_auth_code":   150,   # 인증번호 (카카오/네이버 로그인, 앱 가입)
    "F_account_num": 150,   # 계좌번호 (중고거래, 환불, 더치페이)
    "G_card_pay":    100,   # 카드번호+유효기간 (전화 주문 결제)
    "H_id_otp":      150,   # 신분증+OTP (비대면 서비스, 은행 앱 오류)
    "I_ownership":   150,   # 명의+환급 (차량/부동산, 세금/보험 환급)
}

GROUP_CONFIG = {
    "A_identity": {
        "desc": "주민번호/생년월일/신분 확인이 나오는 정상 본인확인 대화",
        "seeds": [
            "네, 고객님, 성함과 주민번호 앞자리를 말씀해 주시면 합격자 명단에서 확인 후 안내 도와 드리겠습니다.",
            "접수 확인을 위해 생년월일 여섯 자리만 말씀해 주시겠어요? 예약자 명단에서 찾아드릴게요.",
            "통신사 고객센터입니다. 본인 확인을 위해 주민번호 앞 여섯 자리 부탁드립니다.",
            "병원 예약 확인해드릴게요. 생년월일이랑 성함 말씀해 주시겠어요?",
            "보험 가입 여부 확인을 위해 주민번호 앞자리 입력해 주시면 됩니다.",
        ],
        "instruction": """주민번호·생년월일·신분증이 나오지만 완전히 정상적인 상황의 대화를 생성하세요.

정상 상황 예시:
- 시험·수강 합격자 명단 확인
- 병원 접수·예약 확인
- 통신사 명의·요금 확인
- 보험·연금 가입 여부 확인
- 공공기관 민원 본인 확인
- 도서관·시설 회원 확인
- 택배 수령 본인 확인

반드시 지킬 사항:
- 기관 사칭 없음 (수사기관·금감원 절대 불가)
- 금전 요구 없음
- 개인정보를 악의적으로 수집하려는 의도 없음
- 일상적인 고객서비스 맥락""",
    },
    "B_banking": {
        "desc": "이체/비밀번호가 나오는 정상 인터넷뱅킹 문의 대화",
        "seeds": [
            "지금 이체하려고 하는데 비밀번호가 생각이 안 나네요. 어떻게 하면 좋을까요.",
            "인터넷뱅킹 비밀번호를 잊어버렸는데 재설정 어떻게 하나요?",
            "이체 한도가 초과됐다고 뜨는데, 한도 올리려면 어떻게 해야 하나요?",
            "공과금 자동이체 해지하고 싶은데 어떻게 하면 되나요?",
            "이체했는데 상대방 계좌에 안 들어갔다고 하는데 확인 가능한가요?",
        ],
        "instruction": """이체·비밀번호·계좌가 나오지만 완전히 정상적인 인터넷뱅킹 문의 대화를 생성하세요.

정상 상황 예시:
- 인터넷뱅킹 비밀번호 분실·재설정 문의
- 이체 한도 초과·조정 문의
- 공과금·보험료·등록금 자동이체 설정·해지
- 이체 오류·지연 확인 문의
- OTP 기기 분실·재발급 문의
- 카드 결제 비밀번호 변경 문의
- 계좌 이체 수수료 문의

반드시 지킬 사항:
- 사기 의도 없음 (정상 금융기관 고객센터 응대)
- 기관 사칭 없음
- 고객이 자발적으로 문의하는 상황
- 자연스러운 고객-상담원 대화체""",
    },
    "C_card": {
        "desc": "계좌/카드 관련 정상 금융 문의 대화",
        "seeds": [
            "카드를 잃어버렸는데 분실 신고 어떻게 하나요?",
            "체크카드 한도가 얼마인지 확인하고 싶어요.",
            "통장 잔액이 갑자기 줄었는데 확인해 주실 수 있나요?",
            "신용카드 명세서에 모르는 결제 내역이 있어요.",
            "예금 만기일이 언제인지 알 수 있을까요?",
        ],
        "instruction": """계좌·카드·예금·통장이 나오지만 완전히 정상적인 금융 문의 대화를 생성하세요.

정상 상황 예시:
- 카드 분실 신고·재발급
- 계좌 잔액·거래내역 확인
- 예금·적금 만기·이율 문의
- 카드 한도·연회비 문의
- 의심 거래 내역 확인 요청
- 통장 개설·해지 문의
- 해외 결제·환전 문의
- ATM 출금 오류 문의

반드시 지킬 사항:
- 고객이 먼저 문의하는 상황 (피싱처럼 상대방이 먼저 요구하지 않음)
- 정상 금융기관 고객센터 응대
- 사기 의도·압박 없음""",
    },
    "D_loan": {
        "desc": "대출/상환 관련 정상 금융 상담 대화",
        "seeds": [
            "주택담보대출 금리가 요즘 어떻게 되나요? 상담 받고 싶어요.",
            "대출 상환 날짜를 바꾸고 싶은데 어떻게 신청하나요?",
            "신용대출 한도가 얼마나 나오는지 알아볼 수 있을까요?",
            "대출 중도상환하면 수수료가 있나요?",
            "전세자금대출 신청 조건이 어떻게 되나요?",
        ],
        "instruction": """대출·상환·금리·한도가 나오지만 완전히 정상적인 금융 상담 대화를 생성하세요.

정상 상황 예시:
- 주택담보·신용·전세자금 대출 문의
- 대출 금리·한도·조건 상담
- 대출 상환일·방법 변경 문의
- 중도상환 수수료 문의
- 대출 연장·만기 문의
- 정부 지원 대출 신청 절차 문의
- 카드론·마이너스 통장 한도 문의

반드시 지킬 사항:
- 정상적인 금융기관 대출 상담 맥락
- 선입금·수수료 사기 요소 없음
- 고객이 자발적으로 문의하는 상황
- 압박·공포 조성 없음""",
    },
    "E_auth_code": {
        "desc": "인증번호가 나오는 정상 앱/서비스 본인인증 대화",
        "seeds": [
            "카카오 로그인 인증번호가 문자로 왔는데 어디에 입력하면 되나요?",
            "네이버 2단계 인증번호가 안 와요. 어떻게 해야 하나요?",
            "은행 앱 설치하고 인증번호 받았는데 계속 오류가 나요.",
            "휴대폰 본인인증 인증번호 입력하는 칸이 어디 있나요?",
            "문자로 인증번호 받았는데 유효시간이 지나버렸어요.",
        ],
        "instruction": """인증번호·OTP·문자인증이 나오지만 완전히 정상적인 앱/서비스 인증 대화를 생성하세요.

정상 상황 예시:
- 카카오·네이버·구글 로그인 2단계 인증
- 은행·금융 앱 설치 후 첫 인증
- 쇼핑몰·플랫폼 회원가입 본인인증
- 휴대폰 번호 변경 후 재인증
- 인증번호 미수신·만료 오류 문의
- 앱 업데이트 후 재로그인 인증

반드시 지킬 사항:
- 사용자가 먼저 서비스를 이용하려다 인증이 필요한 상황
- 타인에게 인증번호를 요구하거나 전달하는 내용 없음
- 일반적인 서비스 이용 맥락""",
    },
    "F_account_num": {
        "desc": "계좌번호가 나오는 정상 환불/더치페이/중고거래 대화",
        "seeds": [
            "환불받을 계좌번호 알려드릴게요. 국민은행 123-456-789012 이준혁입니다.",
            "더치페이 정산할게요. 제 카카오뱅크 계좌번호 보내드릴까요?",
            "중고나라 거래인데요, 입금하실 계좌번호 알려드릴게요.",
            "보증금 돌려받을 계좌번호 적어드릴게요. 어디 은행으로 받으시겠어요?",
            "공연 취소됐는데 환불 계좌번호 어디로 알려드려야 하나요?",
        ],
        "instruction": """계좌번호가 나오지만 완전히 정상적인 환불·송금·정산 대화를 생성하세요.

정상 상황 예시:
- 쇼핑몰·앱 환불 계좌 등록
- 친구·동료 더치페이 정산
- 중고거래 입금 계좌 안내
- 월세·보증금 반환 계좌 확인
- 공연·행사 취소 환불 처리
- 회사 경비 정산 계좌 제출
- 알바비·프리랜서 대금 수령 계좌

반드시 지킬 사항:
- 상대방이 자발적으로 계좌번호를 요청하거나 제공하는 상황
- 피싱처럼 일방적으로 계좌번호를 요구하는 상황 없음
- 일상적인 거래·환불 맥락""",
    },
    "G_card_pay": {
        "desc": "카드번호/유효기간이 나오는 정상 전화 결제 대화",
        "seeds": [
            "전화 주문할게요. 카드번호 불러드릴게요. 1234-5678-9012-3456이고요.",
            "호텔 예약 보증용으로 카드번호랑 유효기간 알려드릴게요.",
            "카드 유효기간이 이번 달까지인데 결제가 되나요?",
            "신용카드 뒷면 세 자리 CVC 번호가 뭔지 여쭤봐도 될까요? 결제 때 필요해서요.",
            "해외 직구하려는데 카드 유효기간을 어떻게 입력해야 하나요?",
        ],
        "instruction": """카드번호·유효기간·CVC가 나오지만 완전히 정상적인 전화결제·쇼핑 대화를 생성하세요.

정상 상황 예시:
- 전화로 음식·상품 주문 결제
- 호텔·렌터카 예약 보증 카드 등록
- 해외 직구 카드 정보 입력 문의
- 카드 유효기간 만료 전 갱신 문의
- 정기결제 카드 변경 문의
- 구독 서비스 결제 카드 등록

반드시 지킬 사항:
- 고객이 먼저 결제를 원해서 카드 정보를 제공하는 상황
- 일방적으로 카드 정보를 캐내려는 시도 없음
- 정상적인 상거래 맥락""",
    },
    "H_id_otp": {
        "desc": "신분증/OTP가 나오는 정상 비대면 서비스·은행 앱 오류 대화",
        "seeds": [
            "비대면 계좌 개설하려고 신분증 앞면 촬영했는데 계속 실패해요.",
            "OTP 단말기 배터리가 다 됐는데 재발급 어떻게 받나요?",
            "신분증 사진이 흐리게 찍혀서 앱에서 인식을 못 하는데 어떻게 해야 하나요?",
            "스마트OTP 앱 설치했는데 등록이 안 돼요. 도와주실 수 있나요?",
            "운전면허증으로 신분증 인증 되나요? 주민등록증이 없어서요.",
        ],
        "instruction": """신분증·OTP가 나오지만 완전히 정상적인 비대면 서비스 이용·오류 문의 대화를 생성하세요.

정상 상황 예시:
- 비대면 계좌 개설 시 신분증 촬영 오류
- OTP 단말기 분실·배터리 방전 재발급
- 스마트OTP 앱 등록·오류 문의
- 신분증 종류별 인증 가능 여부 문의
- 모바일 운전면허증 사용 문의
- 해외 체류 중 신분 확인 방법 문의

반드시 지킬 사항:
- 고객이 직접 서비스를 이용하려는 상황
- 타인에게 신분증 정보를 넘기는 상황 없음
- 정상적인 금융·공공 서비스 이용 맥락""",
    },
    "I_ownership": {
        "desc": "명의/환급이 나오는 정상 부동산·차량·세금 환급 대화",
        "seeds": [
            "차량 명의 이전하려고 하는데 필요한 서류가 뭔가요?",
            "건강보험료 환급 신청을 하고 싶은데 어떻게 해야 하나요?",
            "부모님 명의 부동산을 제 명의로 바꾸려면 어떤 절차가 필요한가요?",
            "세금 환급금 조회를 어디서 할 수 있나요? 국세청 홈택스인가요?",
            "통신사 명의 변경 절차가 어떻게 되나요?",
        ],
        "instruction": """명의·환급이 나오지만 완전히 정상적인 부동산·차량·세금·보험 관련 대화를 생성하세요.

정상 상황 예시:
- 차량 명의 이전·매매 절차 문의
- 부동산 명의 변경·증여 절차 문의
- 건강보험료·고용보험 환급 신청
- 국세청 세금 환급 조회·신청
- 통신사·금융 서비스 명의 변경
- 보험금 환급·만기 환급 문의
- 과오납 세금 환급 문의

반드시 지킬 사항:
- 고객이 먼저 문의하는 상황
- 명의도용·사기 의도 없음
- 정상적인 행정·금융 절차 맥락
- 압박·공포 조성 없음""",
    },
}


# ── API 클라이언트 ────────────────────────────────────────────
def get_client() -> OpenAI:
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            print("✅ Colab Secrets에서 API 키 로드")
            return OpenAI(api_key=key)
    except Exception:
        pass
    key = os.getenv("OPENAI_API_KEY") or input("OpenAI API Key를 입력하세요: ").strip()
    return OpenAI(api_key=key)


def chat(client: OpenAI, prompt: str, temperature: float = 0.9) -> dict | None:
    for attempt in range(RETRY_LIMIT):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [retry {attempt+1}/{RETRY_LIMIT}] {e} — {wait}s 대기")
            time.sleep(wait)
    return None


def generate_batch(client: OpenAI, group_key: str, seed: str, n: int) -> list[str]:
    cfg = GROUP_CONFIG[group_key]
    prompt = f"""다음은 {cfg['desc']}의 예시입니다.

[참고 텍스트]
{seed}

{cfg['instruction']}

위 조건에 맞는 정상 통화 발화 {n}개를 생성하세요.

규칙:
- 각 텍스트는 충분히 다른 상황·표현 사용
- 실제 통화처럼 자연스러운 구어체
- 길이: 50~300자
- 라벨은 반드시 [0, 0, 0] (정상) — 기관사칭·금전요구·개인정보 수집 의도 없음

JSON: {{"samples": [{{"text": "..."}}]}} 형식으로 {n}개 응답"""

    result = chat(client, prompt)
    if result is None:
        return []
    return [s["text"] for s in result.get("samples", []) if isinstance(s, dict) and s.get("text")]


def main() -> None:
    client = get_client()

    results: list[dict] = []
    if OUTPUT_PATH.exists():
        results = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
        print(f"📂 기존 진행 결과 로드: {len(results)}개 (이어서 진행)")

    from collections import Counter
    current_by_group = Counter(d.get("source", "") for d in results)

    total_target = sum(QUOTA.values())
    print(f"\n=== Hard Negative 정상 데이터 생성 시작 (목표 {total_target}개) ===")

    for group_key, target in QUOTA.items():
        already = current_by_group.get(group_key, 0)
        need    = target - already
        cfg     = GROUP_CONFIG[group_key]

        if need <= 0:
            print(f"  [{group_key}] 이미 완료 ({already}/{target})")
            continue

        print(f"\n  [{group_key}] {need}개 필요 ({already}/{target})")
        print(f"  → {cfg['desc']}")

        generated = 0
        seed_idx  = 0
        pbar      = tqdm(total=need, desc=f"  {group_key}")

        while generated < need:
            seed    = cfg["seeds"][seed_idx % len(cfg["seeds"])]
            batch_n = min(BATCH_SIZE, need - generated)

            texts = generate_batch(client, group_key, seed, batch_n)
            for text in texts:
                results.append({
                    "text":   text,
                    "label":  [0, 0, 0],
                    "source": group_key,
                })
            generated += len(texts)
            seed_idx  += 1
            pbar.update(len(texts))
            time.sleep(SLEEP_SEC)

            if len(results) % 50 == 0:
                OUTPUT_PATH.write_text(
                    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
                )
                print(f"\n  💾 중간 저장: {len(results)}개")

        pbar.close()

    OUTPUT_PATH.write_text(
        json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    dist = Counter(d.get("source", "?") for d in results)
    print(f"\n{'='*55}")
    print(f"  ✅ 총 {len(results)}개 생성 완료 (label=[0,0,0])")
    print(f"  저장: {OUTPUT_PATH}")
    print(f"{'='*55}")
    print("\n그룹별 분포:")
    for grp, cnt in dist.items():
        print(f"  {grp:20s}: {cnt}개")
    print("\n다음 단계:")
    print("  train_colab.ipynb의 SOURCE_FILES에 아래 경로 추가:")
    print("  DATA_DIR / 'raw/normal_hard_negative.json'")

In [ ]:
# ── 셀 3: 실행 ────────────────────────────────────────────────
main()

## 생성 완료 후 — train_colab.ipynb 수정 방법

`train_colab.ipynb`의 `SOURCE_FILES` 리스트에 아래 한 줄 추가:

```python
SOURCE_FILES: list[Path] = [
    DATA_DIR / "v5/phishing_augmented_data.json",
    DATA_DIR / "raw/normal_tts.json",
    DATA_DIR / "raw/normal_callcenter.json",
    DATA_DIR / "raw/callcenter_finance.json",
    DATA_DIR / "raw/normal_hard_negative.json",  # ← 추가
]
```

그리고 `OUTPUT_DIR`을 새 버전명으로 변경:
```python
OUTPUT_DIR = DRIVE_BASE / "models/koelectra-finetuned-v6"
```